# Amplitude Modulation: Frequency Shifting by Multiplication

*Python/Colab conversion of the MATLAB script `modulate.m`*
(from **Software Receiver Design**, Johnson, Sethares & Klein).

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/oalnaseri/CommSystems_Course/blob/main/02_modulate.ipynb)

> **How to run:** Click the **Open in Colab** badge above , then run each cell top-to-bottom with `Shift + Enter`, or use **Runtime -> Run all**.

**The idea:** multiplying a signal by a cosine oscillator *shifts* its spectrum. Here a **100 Hz** input times a **1000 Hz** oscillator produces energy at the **sum (1100 Hz)** and **difference (900 Hz)** frequencies — the basis of AM radio.

No installation is needed — `numpy`, `matplotlib` and `plotly` are pre-installed in Colab.

## 1. Setup

Import the numerical, plotting and interactive-plotting libraries.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Interactive plotting (pre-installed in Colab). Install locally if missing.
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'plotly'])
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

%matplotlib inline
plt.rcParams['figure.figsize'] = (9, 6)

## 2. The `plotspec` helper function (with optional zoom)

A helper that plots both the
**time-domain waveform** and its **magnitude spectrum** (via the FFT). We re-implement it
in Python and add an optional **`flim`** zoom argument for the frequency axis:
* `plotspec(x, Ts)` → full spectrum
* `plotspec(x, Ts, flim=1500)` → zoom to ±1500 Hz
* `plotspec(x, Ts, flim=(800, 1200))` → zoom to a custom range

In [ ]:
def plotspec(x, Ts, flim=None):
    """Plot the waveform and magnitude spectrum of signal x.

    Parameters
    ----------
    x    : 1-D array of signal samples.
    Ts   : sample interval in seconds (Ts = 1/sample_rate).
    flim : optional frequency-axis zoom for the spectrum.
           * None            -> full spectrum (default)
           * a number f      -> zoom to (-f, +f) Hz
           * a tuple (lo, hi)-> zoom to (lo, hi) Hz
    """
    x = np.asarray(x)
    N = len(x)
    t = Ts * np.arange(1, N + 1)            # time vector
    ssf = np.arange(-N/2, N/2) / (Ts * N)   # frequency vector (Hz)
    fxs = np.fft.fftshift(np.fft.fft(x))   # centered FFT

    fig, (ax1, ax2) = plt.subplots(2, 1)
    ax1.plot(t, x)
    ax1.set_xlabel('seconds'); ax1.set_ylabel('amplitude')
    ax1.set_title('Waveform (time domain)'); ax1.grid(True)

    ax2.plot(ssf, np.abs(fxs))
    ax2.set_xlabel('frequency (Hz)'); ax2.set_ylabel('magnitude')
    ax2.grid(True)

    if flim is not None:
        if np.isscalar(flim):
            ax2.set_xlim(-flim, flim)
            ax2.set_title(f'Magnitude spectrum (zoomed to ±{flim} Hz)')
        else:
            ax2.set_xlim(flim[0], flim[1])
            ax2.set_title(f'Magnitude spectrum (zoomed to {flim[0]}–{flim[1]} Hz)')
    else:
        ax2.set_title('Magnitude spectrum (full)')

    fig.tight_layout(); plt.show()

## 3. Interactive version: `plotspec_interactive`

Lets you **zoom the plot themselves** — no code edits needed:
- **Box-zoom:** click and drag a rectangle
- **Pan:** switch to the pan tool and drag
- **Hover:** point at a spike to read its exact frequency & magnitude
- **Reset:** double-click

In [ ]:
def plotspec_interactive(x, Ts, flim=None, title=''):
    """Interactive (drag-to-zoom) waveform + magnitude spectrum using Plotly."""
    x = np.asarray(x)
    N = len(x)
    t = Ts * np.arange(1, N + 1)
    ssf = np.arange(-N/2, N/2) / (Ts * N)
    fxs = np.fft.fftshift(np.fft.fft(x))

    fig = make_subplots(rows=2, cols=1,
                        subplot_titles=('Waveform (time domain)',
                                        'Magnitude spectrum'))
    fig.add_trace(go.Scatter(x=t, y=x, mode='lines', name='waveform'),
                  row=1, col=1)
    fig.add_trace(go.Scatter(x=ssf, y=np.abs(fxs), mode='lines',
                             name='|spectrum|'), row=2, col=1)
    fig.update_xaxes(title_text='seconds', row=1, col=1)
    fig.update_yaxes(title_text='amplitude', row=1, col=1)
    fig.update_xaxes(title_text='frequency (Hz)', row=2, col=1)
    fig.update_yaxes(title_text='magnitude', row=2, col=1)
    if flim is not None:
        lo, hi = (-flim, flim) if np.isscalar(flim) else (flim[0], flim[1])
        fig.update_xaxes(range=[lo, hi], row=2, col=1)
    fig.update_layout(height=650, showlegend=False, title_text=title,
                      hovermode='x unified')
    fig.show()

## 4. Create the signals

* **`x`** — the message/input, a 100 Hz cosine.
* **`cmod`** — the carrier/oscillator, a 1000 Hz cosine.
* **`y`** — their point-by-point product (this is the modulation step).

In [ ]:
time = 0.5          # length of time (seconds)
Ts = 1/10000        # sampling interval  ->  10 kHz sample rate
t = np.arange(Ts, time + Ts, Ts)     # time vector: Ts:Ts:time

fc = 1000                            # carrier / oscillator frequency (Hz)
cmod = np.cos(2*np.pi*fc*t)          # oscillator: cos of freq fc

fi = 100                             # input message frequency (Hz)
x = np.cos(2*np.pi*fi*t)             # input: cos of freq fi

y = cmod * x                         # multiply input by oscillator (element-wise)

print(f'sample rate = {1/Ts:.0f} Hz, N = {len(t)} samples')

## 5. Oscillator `cmod` — spectrum 

The 1000 Hz carrier shows spikes at **±1000 Hz**.

In [ ]:
plotspec(cmod, Ts)                       # full spectrum of the oscillator

**interactive** to ±1500 Hz

In [ ]:
plotspec_interactive(cmod, Ts, flim=1500, title='Oscillator cmod (1000 Hz)')

## 6. Input `x` — spectrum 

The 100 Hz message shows spikes at **±100 Hz**.

In [ ]:
plotspec(x, Ts)                          # full spectrum of the input

**interactive** to ±500 Hz

In [ ]:
plotspec_interactive(x, Ts, flim=500, title='Input x (100 Hz)')

## 7. Output `y = cmod · x` — spectrum 

This is the key result. Multiplying in time **shifts** the input up and down by the carrier
frequency, so the 100 Hz energy moves to:

$$f_c \pm f_i = 1000 \pm 100 = \{900\text{ Hz},\; 1100\text{ Hz}\}$$

You'll see four spikes at **±900 Hz** and **±1100 Hz** — and *nothing* at the original 100 Hz.

In [ ]:
plotspec(y, Ts)                          # full spectrum of the output

**Zoomed-in** to 800–1200 Hz to see the sum/difference pair clearly, then **interactive**:

In [ ]:
plotspec(y, Ts, flim=(800, 1200))

In [ ]:
plotspec_interactive(y, Ts, flim=(800, 1200), title='Output y = cmod * x (900 & 1100 Hz)')

## 8. Side-by-side comparison 

This stacked the three magnitude spectra on one figure so the
frequency shift is obvious at a glance.


In [ ]:
N = len(x)                               # length of the signal
ssf = (np.arange(-N/2, N/2)) / (Ts * N)  # frequency vector (Hz)

fx    = np.fft.fftshift(np.fft.fft(x))     # input spectrum
fcmod = np.fft.fftshift(np.fft.fft(cmod))  # oscillator spectrum
fy    = np.fft.fftshift(np.fft.fft(y))     # output spectrum

fig, axes = plt.subplots(3, 1, figsize=(9, 8))
axes[0].plot(ssf, np.abs(fx))
axes[0].set_xlabel('magnitude spectrum at input (100 Hz)')
axes[1].plot(ssf, np.abs(fcmod))
axes[1].set_xlabel('magnitude spectrum of the oscillator (1000 Hz)')
axes[2].plot(ssf, np.abs(fy))
axes[2].set_xlabel('magnitude spectrum at output (900 & 1100 Hz)')
for ax in axes:
    ax.set_xlim(-1500, 1500)             # zoom so the shift is easy to see
    ax.grid(True)
fig.tight_layout(); plt.show()

## 9. Notes & tips

- **The takeaway:** multiplying by a cosine of frequency $f_c$ copies the input spectrum to $\pm f_c$. A tone at $f_i$ becomes tones at $f_c \pm f_i$ (here 900 & 1100 Hz). This is exactly how AM transmitters move audio up to a radio carrier.
- **Try it yourself:** change `fc` or `fi` and re-run — watch the output spikes move to the new sum and difference frequencies.